# RL + PINN (Complete Solution)

This notebook completes all parts:
- **Part I:** PPO on HalfCheetah-v4
- **Part II:** PINN velocity model (includes missing `extract_velocity_from_rollouts_fixed`)
- **Part III:** Physics-regularized PPO bonus

Set `QUICK_RUN = True` for a shorter local demo (~2–5 min). Set `QUICK_RUN = False` for full lab training (~1M steps).


In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "glfw")  # use "egl" on headless Linux/Colab

# Install on Colab only (safe no-op locally if already installed)
try:
    import google.colab  # noqa: F401
    get_ipython().system('pip install -q "gymnasium[mujoco]" "imageio[ffmpeg]" tensorboard tqdm')
except Exception:
    pass


In [ ]:
import gymnasium as gymn
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
from torch.utils.tensorboard import SummaryWriter
from copy import deepcopy
from tqdm import trange
import imageio
import matplotlib.pyplot as plt

# ===================== CONFIG =====================
QUICK_RUN = True          # False -> full 1M-step PPO training
TOTAL_STEPS = 50_000 if QUICK_RUN else 1_000_000
UPDATE_EVERY = 2048 if QUICK_RUN else 4096
BONUS_STEPS = 30_000 if QUICK_RUN else 200_000
LAMBDA_REGS = [0.0, 0.001, 0.01, 0.1]
ENV_ID = "HalfCheetah-v4"
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)
print("Gymnasium:", gymn.__version__)


In [ ]:
# ===================== PART I: PPO CORE =====================

class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.mu = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim))

    def forward(self, x):
        z = self.net(x)
        return self.mu(z), self.log_std.exp()


class Critic(nn.Module):
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class RolloutBuffer:
    def __init__(self):
        self.states, self.actions, self.logprobs = [], [], []
        self.values, self.rewards, self.dones = [], [], []

    def add(self, state, action, logprob, value, reward, done):
        self.states.append(state)
        self.actions.append(action)
        self.logprobs.append(logprob)
        self.values.append(value)
        self.rewards.append(reward)
        self.dones.append(done)

    def clear(self):
        self.__init__()


def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        mask = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values[t + 1] * mask - values[t]
        gae = delta + gamma * lam * mask * gae
        advantages[t] = gae
    returns = advantages + np.array(values[:-1])
    return advantages, returns


class PPO:
    def __init__(self, actor, critic, clip_eps=0.2, gamma=0.99, lam=0.95,
                 lr=3e-4, epochs=10, entropy_coef=0.01):
        self.actor = actor
        self.critic = critic
        self.gamma = gamma
        self.lam = lam
        self.clip_eps = clip_eps
        self.epochs = epochs
        self.entropy_coef = entropy_coef
        self.optimizer = optim.Adam(
            list(actor.parameters()) + list(critic.parameters()), lr=lr
        )

    def update(self, buffer: RolloutBuffer, next_value: float):
        values_ext = buffer.values + [next_value]
        advantages, returns = compute_gae(
            buffer.rewards, values_ext, buffer.dones, self.gamma, self.lam
        )
        advantages = torch.as_tensor(advantages, dtype=torch.float32)
        returns = torch.as_tensor(returns, dtype=torch.float32)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        states = torch.as_tensor(buffer.states, dtype=torch.float32)
        actions = torch.as_tensor(buffer.actions, dtype=torch.float32)
        old_logprobs = torch.as_tensor(buffer.logprobs, dtype=torch.float32)

        eps = 1e-6
        metrics = {}
        for _ in range(self.epochs):
            mu, std = self.actor(states)
            dist = Normal(mu, std)
            actions_clipped = actions.clamp(-1 + eps, 1 - eps)
            u = torch.atanh(actions_clipped)
            logprobs = dist.log_prob(u) - torch.log(1 - actions_clipped.pow(2) + eps)
            logprobs = logprobs.sum(dim=-1)
            entropy = dist.entropy().sum(dim=-1).mean()
            values_pred = self.critic(states)
            ratios = torch.exp(logprobs - old_logprobs)
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.clip_eps, 1 + self.clip_eps) * advantages
            actor_loss = -torch.min(surr1, surr2).mean()
            value_loss = (returns - values_pred).pow(2).mean()
            approx_kl = (old_logprobs - logprobs).mean()
            loss = actor_loss + 0.5 * value_loss - self.entropy_coef * entropy
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            metrics = {
                "loss": loss.item(),
                "actor_loss": actor_loss.item(),
                "value_loss": value_loss.item(),
                "entropy": entropy.item(),
                "approx_kl": approx_kl.item(),
            }
        return metrics


def ppo_act(obs, actor, critic):
    obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        mu, std = actor(obs_t)
        dist = Normal(mu, std)
        u = dist.rsample()
        action = torch.tanh(u)
        logp = dist.log_prob(u) - torch.log(1 - action.pow(2) + 1e-6)
        logp = logp.sum(dim=-1)
        value = critic(obs_t)
    return action.squeeze(0).cpu().numpy(), float(logp.item()), float(value.item())


def eval_action(obs, actor):
    obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        mu, _ = actor(obs_t)
        action = torch.tanh(mu)
    return action.squeeze(0).cpu().numpy()


def get_forward_velocity(env):
    """Root x-velocity from MuJoCo (m/s)."""
    return float(env.unwrapped.data.qvel[0])


In [ ]:
# ===================== PART I: TRAIN PPO =====================

def train_ppo(total_steps=TOTAL_STEPS, update_every=UPDATE_EVERY, log_dir="runs/ppo_halfcheetah"):
    env = gymn.make(ENV_ID)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    actor = Actor(obs_dim, act_dim)
    critic = Critic(obs_dim)
    ppo = PPO(actor, critic)

    writer = SummaryWriter(log_dir=log_dir)
    buffer = RolloutBuffer()

    episode_rewards, moving_avg_rewards = [], []
    best_avg_reward = -np.inf
    best_actor_state = best_critic_state = None

    obs, _ = env.reset(seed=SEED)
    episode_reward = 0.0
    global_episode = 0

    for step in trange(1, total_steps + 1, desc="PPO training"):
        action, logp, value = ppo_act(obs, actor, critic)
        action = np.clip(action, -1.0, 1.0)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        episode_reward += reward
        buffer.add(obs, action, logp, value, reward, done)
        obs = next_obs

        if done:
            global_episode += 1
            episode_rewards.append(episode_reward)
            moving_avg = np.mean(episode_rewards[-20:])
            moving_avg_rewards.append(moving_avg)
            writer.add_scalar("Episode/Reward", episode_reward, global_episode)
            writer.add_scalar("Episode/MovingAvg20", moving_avg, global_episode)
            if moving_avg > best_avg_reward:
                best_avg_reward = moving_avg
                best_actor_state = deepcopy(actor.state_dict())
                best_critic_state = deepcopy(critic.state_dict())
            obs, _ = env.reset()
            episode_reward = 0.0

        if step % update_every == 0:
            with torch.no_grad():
                next_value = critic(torch.as_tensor(obs, dtype=torch.float32)).item()
            metrics = ppo.update(buffer, next_value)
            buffer.clear()
            writer.add_scalar("Loss/Total", metrics["loss"], step)
            writer.add_scalar("Diag/ApproxKL", metrics["approx_kl"], step)

    writer.close()
    env.close()

    torch.save(actor.state_dict(), "ppo_last_actor.pth")
    torch.save(critic.state_dict(), "ppo_last_critic.pth")
    if best_actor_state is not None:
        torch.save(best_actor_state, "ppo_best_actor.pth")
        torch.save(best_critic_state, "ppo_best_critic.pth")

    plt.figure(figsize=(10, 4))
    plt.plot(episode_rewards, alpha=0.35, label="Episode reward")
    plt.plot(moving_avg_rewards, linewidth=2, label="MA(20)")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.title("PPO on HalfCheetah-v4")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Best MA(20): {best_avg_reward:.1f}")
    return obs_dim, act_dim, episode_rewards, moving_avg_rewards

obs_dim, act_dim, episode_rewards, moving_avg_rewards = train_ppo()


## Part II — PINN velocity model

Missing function from the original lab: extract forward velocity from PPO rollouts.


In [ ]:
# ===================== PART II: PINN =====================

def extract_velocity_from_rollouts_fixed(env, actor, n_rollouts=5, max_steps=200):
    """Collect (time, forward_velocity) pairs from deterministic PPO rollouts."""
    t_data, v_data = [], []
    dt = float(env.unwrapped.dt)

    for _ in range(n_rollouts):
        obs, _ = env.reset()
        t = 0.0
        for _ in range(max_steps):
            v = get_forward_velocity(env)
            t_data.append(t)
            v_data.append(v)

            action = eval_action(obs, actor)
            action = np.clip(action, -1.0, 1.0)
            obs, _, terminated, truncated, _ = env.step(action)
            t += dt
            if terminated or truncated:
                break

    return np.asarray(t_data, dtype=np.float32), np.asarray(v_data, dtype=np.float32)


class PINN_Simple(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=3):
        super().__init__()
        layers = [nn.Linear(1, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
        self.alpha = nn.Parameter(torch.tensor(1.0))
        self.c = nn.Parameter(torch.tensor(0.1))

    def forward(self, t):
        return self.net(t)

    def compute_physics_residual(self, t):
        v = self.forward(t)
        dv_dt = torch.autograd.grad(
            v, t, grad_outputs=torch.ones_like(v), create_graph=True, retain_graph=True
        )[0]
        return dv_dt - (self.alpha - self.c * v)

    def get_parameters(self):
        return self.alpha.item(), self.c.item()


def train_pinn_simple(pinn, t_data, v_data, physics_times, epochs=2000,
                      lr=5e-3, lambda_phys=0.2, verbose=True):
    t_tensor = torch.FloatTensor(t_data).unsqueeze(1)
    v_tensor = torch.FloatTensor(v_data).unsqueeze(1)
    t_phys = torch.FloatTensor(physics_times).unsqueeze(1)
    t_phys.requires_grad_(True)

    optimizer = torch.optim.Adam(pinn.parameters(), lr=lr)
    losses = {k: [] for k in ["total", "data", "physics", "alpha", "c"]}
    iterator = trange(epochs, desc="PINN training") if verbose else range(epochs)

    for epoch in iterator:
        optimizer.zero_grad()
        data_loss = torch.mean((pinn(t_tensor) - v_tensor) ** 2)
        physics_loss = torch.mean(pinn.compute_physics_residual(t_phys) ** 2)
        total_loss = data_loss + lambda_phys * physics_loss
        total_loss.backward()
        optimizer.step()

        losses["total"].append(total_loss.item())
        losses["data"].append(data_loss.item())
        losses["physics"].append(physics_loss.item())
        alpha, c = pinn.get_parameters()
        losses["alpha"].append(alpha)
        losses["c"].append(c)

    return losses


def plot_pinn_results(t_data, v_data, pinn, losses):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    t_smooth = np.linspace(t_data.min(), t_data.max(), 500)
    with torch.no_grad():
        v_smooth = pinn(torch.FloatTensor(t_smooth).unsqueeze(1)).numpy().squeeze()

    axes[0].scatter(t_data[::5], v_data[::5], s=4, alpha=0.4, label="data")
    axes[0].plot(t_smooth, v_smooth, "r-", linewidth=2, label="PINN")
    axes[0].set(xlabel="time (s)", ylabel="v (m/s)", title="Velocity fit")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].semilogy(losses["total"], label="total")
    axes[1].semilogy(losses["data"], "--", label="data")
    axes[1].semilogy(losses["physics"], "--", label="physics")
    axes[1].set(xlabel="epoch", ylabel="loss", title="PINN losses")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(losses["alpha"], label="alpha")
    axes[2].plot(losses["c"], label="c")
    axes[2].set(xlabel="epoch", ylabel="value", title="Learned parameters")
    axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    alpha, c = pinn.get_parameters()
    print(f"Learned ODE: dv/dt = {alpha:.4f} - {c:.4f} * v")
    return alpha, c


def run_pinn_experiment(obs_dim, act_dim):
    env = gymn.make(ENV_ID)
    actor = Actor(obs_dim, act_dim)
    actor.load_state_dict(torch.load("ppo_best_actor.pth", map_location="cpu"))
    actor.eval()

    t_data, v_data = extract_velocity_from_rollouts_fixed(env, actor, n_rollouts=5, max_steps=200)
    print(f"Collected {len(t_data)} samples, v in [{v_data.min():.2f}, {v_data.max():.2f}]")

    t_physics = np.linspace(t_data.min(), t_data.max(), 500)
    pinn = PINN_Simple(hidden_dim=128, n_layers=4)
    losses = train_pinn_simple(pinn, t_data, v_data, t_physics, epochs=1500 if QUICK_RUN else 2000)
    alpha, c = plot_pinn_results(t_data, v_data, pinn, losses)

    torch.save(pinn.state_dict(), "pinn_velocity_model.pth")
    env.close()
    return pinn, alpha, c, t_data, v_data

pinn_model, alpha_learned, c_learned, t_data, v_data = run_pinn_experiment(obs_dim, act_dim)


## Part III — Physics-regularized PPO (Bonus)

Regularized reward: $r'_t = r_t - \lambda_{reg} R(t)^2$ where $R(t)$ is the PINN physics residual at episode time $t$.


In [ ]:
# ===================== PART III: PHYSICS-REGULARIZED PPO =====================

def compute_pinn_residual_sq(pinn: PINN_Simple, t_scalar: float) -> float:
    """Physics residual squared from frozen PINN at continuous time t."""
    t = torch.tensor([[t_scalar]], dtype=torch.float32, requires_grad=True)
    v = pinn(t)
    dv_dt = torch.autograd.grad(v, t, grad_outputs=torch.ones_like(v))[0]
    residual = dv_dt - (pinn.alpha - pinn.c * v)
    return float((residual ** 2).item())


def train_ppo_with_physics(pinn, lambda_reg=0.01, total_steps=BONUS_STEPS, update_every=UPDATE_EVERY):
    pinn.eval()
    for p in pinn.parameters():
        p.requires_grad_(False)

    env = gymn.make(ENV_ID)
    dt = float(env.unwrapped.dt)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    actor = Actor(obs_dim, act_dim)
    critic = Critic(obs_dim)
    ppo = PPO(actor, critic)

    buffer = RolloutBuffer()
    episode_rewards = []
    residual_sq_log = []

    obs, _ = env.reset(seed=SEED)
    episode_reward = 0.0
    episode_time = 0.0

    for step in trange(1, total_steps + 1, desc=f"PPO lambda={lambda_reg}"):
        action, logp, value = ppo_act(obs, actor, critic)
        action = np.clip(action, -1.0, 1.0)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        residual_sq = compute_pinn_residual_sq(pinn, episode_time)
        residual_sq_log.append(residual_sq)
        shaped_reward = reward - lambda_reg * residual_sq

        episode_reward += reward
        buffer.add(obs, action, logp, value, shaped_reward, done)
        obs = next_obs
        episode_time += dt

        if done:
            episode_rewards.append(episode_reward)
            obs, _ = env.reset()
            episode_reward = 0.0
            episode_time = 0.0

        if step % update_every == 0:
            with torch.no_grad():
                next_value = critic(torch.as_tensor(obs, dtype=torch.float32)).item()
            ppo.update(buffer, next_value)
            buffer.clear()

    env.close()
    ma20 = np.mean(episode_rewards[-20:]) if len(episode_rewards) >= 20 else np.mean(episode_rewards)
    return {
        "lambda_reg": lambda_reg,
        "episode_rewards": episode_rewards,
        "ma20": ma20,
        "avg_residual_sq": float(np.mean(residual_sq_log)),
    }


def compare_lambda_values(pinn, lambdas=LAMBDA_REGS):
    results = []
    for lam in lambdas:
        print(f"\nTraining with lambda_reg = {lam}")
        results.append(train_ppo_with_physics(pinn, lambda_reg=lam))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for res in results:
        rewards = res["episode_rewards"]
        ma = np.convolve(rewards, np.ones(20) / 20, mode="valid") if len(rewards) >= 20 else rewards
        axes[0].plot(ma, label=f"lambda={res['lambda_reg']}")
    axes[0].set(xlabel="episode", ylabel="reward (MA20)", title="Learning curves")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    lambdas_plot = [r["lambda_reg"] for r in results]
    axes[1].bar([str(l) for l in lambdas_plot], [r["avg_residual_sq"] for r in results], alpha=0.75)
    axes[1].set(xlabel="lambda_reg", ylabel="mean R(t)^2", title="Physics violation during training")
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout(); plt.show()

    print("\nSummary:")
    for r in results:
        print(
            f"  lambda={r['lambda_reg']:.4g} | MA20={r['ma20']:.1f} | "
            f"avg R^2={r['avg_residual_sq']:.6f} | episodes={len(r['episode_rewards'])}"
        )
    return results

bonus_results = compare_lambda_values(pinn_model)


In [ ]:
# Optional: record a short video with the best baseline PPO policy
try:
    from IPython.display import HTML
    import base64

    def record_video(env_id, actor, filename="cheetah_best.mp4", max_steps=400):
        env = gymn.make(env_id, render_mode="rgb_array")
        obs, _ = env.reset(seed=SEED)
        frames = []
        for _ in range(max_steps):
            action = np.clip(eval_action(obs, actor), -1.0, 1.0)
            obs, _, terminated, truncated, _ = env.step(action)
            frames.append(env.render())
            if terminated or truncated:
                break
        env.close()
        imageio.mimsave(filename, frames, fps=30)
        print("Saved", filename)
        return filename

    actor = Actor(obs_dim, act_dim)
    actor.load_state_dict(torch.load("ppo_best_actor.pth", map_location="cpu"))
    vid = record_video(ENV_ID, actor)
    video_b64 = base64.b64encode(open(vid, "rb").read()).decode("ascii")
    HTML(f'<video width="640" controls><source src="data:video/mp4;base64,{video_b64}" type="video/mp4"></video>')
except Exception as e:
    print("Video skipped:", e)
